# Event log exploration

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pm4py

# Get the root of the project
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config').is_dir())
sys.path.insert(0, str(ROOT))

from src.configs import load_config
from src.logs.io import read_log
from src.logs.keys import (
    ACTIVITY_KEY,
    CASE_KEY,
    CASE_OFFSET_KEY,
    EVENT_DELTA_KEY,
    LABEL_KEY,
    MISSING_FEATURE,
    RESOURCE_KEY,
    TIMESTAMP_KEY,
)

# Set global plotting and pandas options
plt.rcParams['figure.figsize'] = (11, 3.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

Pick the dataset here: everything below is driven by its config.

In [ ]:
DATASET = 'bpic2012_a'  # any config/<name>.yaml

# Load the dataset configuration
data_config = load_config(ROOT / 'config' / f'{DATASET}.yaml').data
processed_dir = ROOT / data_config.dir / 'processed'

# Load the full log sorted by case and timestamp, and the train/val/test splits
log = read_log(processed_dir / 'full.csv').sort_values([CASE_KEY, TIMESTAMP_KEY], kind='stable')
splits = {name: read_log(processed_dir / f'{name}.csv') for name in ('train', 'val', 'test')}

# Convert the resource column to string type for all logs
for frame in (log, *splits.values()):
    frame[RESOURCE_KEY] = frame[RESOURCE_KEY].astype('string')

# Group the log by case and extract the trace lengths and traces
cases = log.groupby(CASE_KEY, sort=False)
trace_len = cases.size()
traces = cases[ACTIVITY_KEY].apply(tuple)

log.head()

## Overview

In [ ]:
pd.Series(
    {
        'events': len(log),
        'proposed_cases': len(cases),
        'activities': log[ACTIVITY_KEY].nunique(),
        'resources': log[RESOURCE_KEY].nunique(),
        'unique traces': traces.nunique(),
        'unique traces / cases': round(traces.nunique() / len(traces), 3),
        'mean trace length': round(trace_len.mean(), 3),
        'median trace length': trace_len.median(),
        'min trace length': trace_len.min(),
        'max trace length': trace_len.max(),
        'first event': log[TIMESTAMP_KEY].min(),
        'last event': log[TIMESTAMP_KEY].max(),
        'columns': log.shape[1],
    }
).to_frame('value')

In [ ]:
pd.DataFrame(
    {
        name: {
            'events': len(split),
            'cases': split[CASE_KEY].nunique(),
            'activities': split[ACTIVITY_KEY].nunique(),
            'resources': split[RESOURCE_KEY].nunique(),
            'unique traces': split.groupby(CASE_KEY, sort=False)[ACTIVITY_KEY]
            .apply(tuple)
            .nunique(),
            'first case start': split.groupby(CASE_KEY)[TIMESTAMP_KEY].min().min(),
            'last case start': split.groupby(CASE_KEY)[TIMESTAMP_KEY].min().max(),
        }
        for name, split in splits.items()
    }
)

## Trace length

In [ ]:
print('quantiles of trace length (in events):')
print(trace_len.quantile([0.5, 0.75, 0.9, 0.95, 0.99, 1.0]).to_string())
print()
print(
    f'cases longer than max_seq_len={data_config.max_seq_len}: '
    f'{(trace_len > data_config.max_seq_len).mean():.2%}'
)
print(
    f'events kept when truncating to max_seq_len: '
    f'{trace_len.clip(upper=data_config.max_seq_len).sum() / trace_len.sum():.2%}'
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(trace_len, bins=min(60, int(trace_len.max())))
axes[0].axvline(data_config.max_seq_len, color='red', ls='--', label='max_seq_len')
axes[0].set(xlabel='trace length', ylabel='cases')
axes[0].legend()
axes[1].plot(np.sort(trace_len), np.linspace(0, 1, len(trace_len)))
axes[1].axvline(data_config.max_seq_len, color='red', ls='--')
axes[1].set(xlabel='trace length', ylabel='cumulative share of cases', xscale='log')
plt.tight_layout()

## Activities

In [ ]:
activity_stats = (
    pd.DataFrame(
        {
            'events': log[ACTIVITY_KEY].value_counts(),
            'event share': log[ACTIVITY_KEY].value_counts(normalize=True).round(4),
            'cases': log.groupby(ACTIVITY_KEY)[CASE_KEY].nunique(),
            'case share': (
                log.groupby(ACTIVITY_KEY)[CASE_KEY].nunique() / log[CASE_KEY].nunique()
            ).round(4),
            'as start': cases[ACTIVITY_KEY].first().value_counts(),
            'as end': cases[ACTIVITY_KEY].last().value_counts(),
        }
    )
    .fillna(0)
    .astype({'as start': int, 'as end': int})
    .sort_values('events', ascending=False)
)
activity_stats

In [ ]:
# Get the top 25 activities
top = activity_stats.head(25).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(12, max(3.5, 0.25 * len(top))))
axes[0].barh(top.index, top['events'])
axes[0].set(xlabel='events', title='activity frequency')
axes[1].barh(top.index, top['as start'], label='as start')
axes[1].barh(top.index, top['as end'], left=top['as start'], label='as end')
axes[1].set(xlabel='cases', title='start / end activities')
axes[1].legend()
plt.tight_layout()

In [ ]:
# Assign a relative position to each event in its case, ranging from 0 (first event) to
# 1 (last event)
position = cases.cumcount() / (cases[ACTIVITY_KEY].transform('size') - 1).replace(0, 1)

top_activities = activity_stats.head(15).index
mix = pd.crosstab(
    pd.cut(position, np.linspace(0, 1, 11)), log[ACTIVITY_KEY], normalize='index'
).reindex(columns=top_activities)
mix.plot.area(
    figsize=(11, 4), cmap='tab20', xlabel='relative position in case', ylabel='share of events'
)
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()

## Resources

In [ ]:
resource_events = log[RESOURCE_KEY].value_counts(dropna=False)
resource_stats = pd.DataFrame(
    {
        'events': resource_events,
        'event share': log[RESOURCE_KEY].value_counts(normalize=True, dropna=False).round(4),
        'cases': log.groupby(RESOURCE_KEY, dropna=False)[CASE_KEY].nunique(),
        'activities': log.groupby(RESOURCE_KEY, dropna=False)[ACTIVITY_KEY].nunique(),
    }
)
print(
    f'resources: {len(resource_events)}  |  missing resource: {log[RESOURCE_KEY].isna().mean():.2%}'
)
print(f'top-10 resources cover {resource_events.head(10).sum() / len(log):.2%} of events')
print(
    f'distinct resources per case: mean {cases[RESOURCE_KEY].nunique().mean():.2f}, '
    f'max {cases[RESOURCE_KEY].nunique().max()}'
)
resource_stats.head(25)

In [ ]:
top = resource_stats.head(25).iloc[::-1]
plt.figure(figsize=(12, max(3.5, 0.25 * len(top))))
plt.barh(top.index.astype(str), top['events'])
plt.xlabel('events')
plt.title('resource frequency (top 25)')
plt.tight_layout()

## Time

In [ ]:
delta = log.loc[cases.cumcount() > 0, EVENT_DELTA_KEY]
duration = cases[TIMESTAMP_KEY].max() - cases[TIMESTAMP_KEY].min()
clip = np.percentile(delta, data_config.time_clip_percentile)

print('event delta (minutes, first event of each case excluded)')
print(delta.describe().round(2).to_string())
print(
    f'\nzero deltas: {(delta == 0).mean():.2%}  |  '
    f'p{data_config.time_clip_percentile} clip value: {clip:.1f} min '
    f'({clip / 60 / 24:.2f} days), clipped events: {(delta > clip).mean():.2%}'
)
print(
    f'\ncase duration (days)\n'
    f'{(duration.dt.total_seconds() / 86400).describe().round(2).to_string()}'
)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(np.log10(delta[delta > 0]), bins=60)
axes[0].axvline(np.log10(max(clip, 1e-9)), color='red', ls='--', label='clip')
axes[0].set(xlabel='log10 event delta (min)', ylabel='events')
axes[0].legend()
axes[1].hist(
    (duration.dt.total_seconds() / 86400).clip(
        upper=(duration.dt.total_seconds() / 86400).quantile(0.99)
    ),
    bins=60,
)
axes[1].set(xlabel='case duration (days, clipped at p99)', ylabel='cases')
axes[2].hist(log[CASE_OFFSET_KEY] / 60 / 24, bins=60)
axes[2].set(xlabel='case offset from log start (days)', ylabel='events')
plt.tight_layout()

In [ ]:
starts = cases[TIMESTAMP_KEY].min()
split_cases = {name: set(split[CASE_KEY]) for name, split in splits.items()}
fig, ax = plt.subplots(figsize=(11, 3.5))
for name in splits:
    ax.hist(starts[starts.index.isin(split_cases[name])], bins=60, label=name, alpha=0.8)
ax.set(xlabel='case start', ylabel='cases', title='temporal split')
ax.legend()
plt.tight_layout()

## Event features

In [ ]:
features = data_config.event_features
missing = [f for f in features if f not in log.columns]
print(f'event features: {features}')
if missing:
    print(f'MISSING FROM LOG: {missing}')
features = [f for f in features if f in log.columns]

# `channel` is the rule DatasetDescription applies: the column's dtype, numeric or not. `level` is
# read off the data rather than declared, since the model reads both the same way - a channel on
# every event - and a case-level column is simply one whose value never changes.
# A categorical column carries MISSING_FEATURE where the log had nothing, a numeric one a NaN,
# so `present` is what both are read through below.
present = {f: log[f][log[f].notna() & log[f].ne(MISSING_FEATURE)] for f in features}
per_case = cases[features].nunique().max()  # 1 => constant within a case
pd.DataFrame(
    {
        'channel': [
            'numeric' if pd.api.types.is_numeric_dtype(log[f]) else 'categorical' for f in features
        ],
        'dtype': [log[f].dtype for f in features],
        'level': ['case' if per_case[f] <= 1 else 'event' for f in features],
        'unique': [present[f].nunique() for f in features],
        'missing': [round(1 - len(present[f]) / len(log), 4) for f in features],
        'example values': [list(present[f].unique()[:4]) for f in features],
    },
    index=features,
)

In [ ]:
numeric = [f for f in features if pd.api.types.is_numeric_dtype(log[f]) and log[f].nunique() > 2]
categorical = [f for f in features if f not in numeric]

if numeric:
    display(log[numeric].describe().T.round(3))
    ncols = min(4, len(numeric))
    fig, axes = plt.subplots(
        -(-len(numeric) // ncols), ncols, figsize=(3 * ncols, 2.4 * -(-len(numeric) // ncols))
    )
    # The grid can have more cells than features (rows * cols rounds up), the leftover axes are
    # hidden below.
    for ax, f in zip(np.ravel([axes]), numeric, strict=False):
        values = log[f].dropna()
        ax.hist(values.clip(upper=values.quantile(0.99)), bins=40)
        ax.set_title(f, fontsize=9)
    for ax in np.ravel([axes])[len(numeric) :]:
        ax.axis('off')
    plt.tight_layout()

In [ ]:
for f in categorical:
    counts = log[f].value_counts(dropna=False, normalize=True).round(4)
    print(f'{f}  ({present[f].nunique()} values, {1 - len(present[f]) / len(log):.1%} missing)')
    print(counts.head(8).to_string(), '\n')

In [ ]:
# Label balance per split, at case level
pd.DataFrame(
    {
        name: split.groupby(CASE_KEY)[LABEL_KEY].first().value_counts(normalize=True).round(3)
        for name, split in splits.items()
    }
)

## Trace variants

In [ ]:
variants = traces.value_counts()
coverage = variants.cumsum() / variants.sum()
print(f'unique traces: {len(variants)} over {len(traces)} cases')
print(
    f'singleton variants: {(variants == 1).sum()} ({(variants == 1).mean():.2%} of variants, '
    f'{(variants == 1).sum() / len(traces):.2%} of cases)'
)
for k in (1, 5, 10, 25, 50, 100):
    if k <= len(variants):
        print(f'top {k:>4} variants cover {coverage.iloc[k - 1]:.2%} of cases')

plt.figure(figsize=(11, 3.5))
plt.plot(np.arange(1, len(variants) + 1), coverage.values)
plt.xscale('log')
plt.xlabel('variants (sorted by frequency)')
plt.ylabel('cumulative share of cases')
plt.tight_layout()

pd.DataFrame(
    {
        'cases': variants.head(15).values,
        'share': (variants.head(15) / len(traces)).round(4).values,
        'length': [len(v) for v in variants.head(15).index],
        'trace': [' -> '.join(v)[:120] for v in variants.head(15).index],
    }
)

## Process model

In [ ]:
NOISE_THRESHOLD = 0.2  # higher = simpler model, more behaviour left out
RANKDIR = 'LR'  # 'TB' reads better when the model is very wide

event_log = pm4py.format_dataframe(
    log[[CASE_KEY, ACTIVITY_KEY, TIMESTAMP_KEY]].copy(),
    case_id=CASE_KEY,
    activity_key=ACTIVITY_KEY,
    timestamp_key=TIMESTAMP_KEY,
)
net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(
    event_log, noise_threshold=NOISE_THRESHOLD
)
print(f'{len(net.places)} places, {len(net.transitions)} transitions, {len(net.arcs)} arcs')
pm4py.view_petri_net(net, initial_marking, final_marking, rankdir=RANKDIR)

In [ ]:
pm4py.view_dfg(*pm4py.discover_dfg(event_log), rankdir=RANKDIR)